In [2]:
import random
import matplotlib.pyplot as plt

In [3]:
def generate_instance(n: int):
    random.seed(42)
    objects = set(range(n))
    sets = tuple(frozenset(random.sample(range(n), k=i + 1)) for i in range(n))
    costs =tuple(10*random.random()+(s*random.random())**2 for s in range(n))

    return objects, sets, costs

In [4]:
def fitness(state: tuple[bool, ...], objects: set, sets: tuple, costs: tuple, penalty_weight: float):
    total_cost = 0.0
    covered = set()

    for idx, selected in enumerate(state):
        if selected:
            total_cost += costs[idx]
            covered.update(sets[idx])

    uncovered_count = len(objects - covered)
    penalty = uncovered_count * penalty_weight

    return -(total_cost + penalty), uncovered_count == 0

In [5]:
def tweak(state: tuple[bool, ...], n: int, p_swap: float = 0.35) -> tuple[bool, ...]:
    new_state = list(state)
    trues = [i for i, v in enumerate(new_state) if v]
    falses = [i for i, v in enumerate(new_state) if not v]

    #Swap 2 bit
    if random.random() < p_swap and trues and falses:
        new_state[random.choice(trues)] = False
        new_state[random.choice(falses)] = True
        return tuple(new_state)

    #1-bit flip standard
    flip_idx = random.randrange(n)
    new_state[flip_idx] = not new_state[flip_idx]
    return tuple(new_state)

In [6]:
def hill_climbing(objects: set, sets: tuple, costs: tuple, max_evals_without_improvement: int):
    n = len(objects)
    penalty_weight = sum(costs) + 1.0

    current_state = tuple(random.choice([True, False]) for _ in range(n))
    current_fit, current_valid = fitness(current_state, objects, sets, costs, penalty_weight)

    evals = 0
    steps = 0

    while evals < max_evals_without_improvement:
        neighbor = tweak(current_state, n)
        neighbor_fit, neighbor_valid = fitness(neighbor, objects, sets, costs, penalty_weight)
        steps += 1

        if neighbor_fit > current_fit:
            current_state = neighbor
            current_fit = neighbor_fit
            current_valid = neighbor_valid
            evals = 0
        else:
            evals += 1

    real_cost = (sum(costs[i] for i, sel in enumerate(current_state) if sel) if current_valid else None)

    return current_state, real_cost, current_valid

In [ ]:
instances_n = list(range(1, 101))
costs_results = []
num_sets_used = []
solutions_log = []
valid_count = 0

for n in instances_n:
    objects, sets, costs = generate_instance(n)
    best_state, cost, is_valid = hill_climbing(objects, sets, costs, max_evals_without_improvement=200)

    if is_valid:
        valid_count += 1
        costs_results.append(cost)
        num_sets_used.append(sum(best_state))
        solutions_log.append(best_state)
    else:
        costs_results.append(None)
        num_sets_used.append(0)
        solutions_log.append(None)

    print(
        f"Instance S_{n:3d} resolved | Valid: {is_valid} | "
        f"Fitness: {-cost:6.2f} | Set Covered: {sum(best_state)}"
    )

print(f"\nCompleted! Valid Instances: {valid_count}/100")

TypeError: hill_climbing() got an unexpected keyword argument 'restarts'